# Regime-Switching Risk-Parity Crypto Index Vault
## Complete ML Pipeline — VS Code Colab Extension (T4 GPU)

**IFTE0007 Individual Coursework** | UCL MSc Digital Finance & Banking

This notebook runs the full compute-heavy ML pipeline:
1. Data fetching and preprocessing
2. GARCH-DCC covariance estimation
3. Bayesian HMM regime detection
4. SAC deep RL training (GPU-accelerated)
5. Walk-forward backtesting
6. Monte Carlo stress testing
7. Final weight computation

**Runtime: ~4-6 hours total** (SAC training is the bottleneck)

---

**Architecture:**
```
Off-chain (Python)              On-chain (Solidity)
─────────────────               ──────────────────
GARCH-DCC ──┐                   RiskParityVault.sol (ERC-4626)
HMM ────────┼── Ensemble ──────► commitWeights(merkleRoot)
SAC RL ─────┘   (CVaR opt)      [timelock 1hr]
                                 executeWeights(w[], proof[])
```

In [ ]:
# ============================================================
# Cell 1: Environment Setup — Install all dependencies
# ============================================================
# Install core ML/DL packages for the full pipeline
!pip install -q \
    arch==7.1.0 \
    hmmlearn==0.3.2 \
    stable-baselines3==2.3.2 \
    gymnasium==0.29.1 \
    cvxpy==1.5.3 \
    ccxt==4.4.26 \
    yfinance==0.2.43 \
    torch \
    pandas \
    numpy \
    scipy \
    scikit-learn \
    matplotlib \
    seaborn \
    tqdm \
    web3 \
    eth-abi \
    pyyaml

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. SAC training will be slow on CPU.")
    print("Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================================
# Cell 2: Project Setup (VS Code Colab Extension)
# ============================================================
# When using the VS Code Colab extension, the notebook file is local
# but code executes on Google's remote runtime. We clone from GitHub
# to make all project files available on the Colab runtime.

import os

REPO_URL = "https://github.com/abailey81/Regime-Switching-Risk-Parity-Crypto-Index-Vault.git"
PROJECT_DIR = "/content/Regime-Switching-Risk-Parity-Crypto-Index-Vault"

if not os.path.exists(PROJECT_DIR):
    print(f"Cloning repository from GitHub...")
    !git clone {REPO_URL}
else:
    print(f"Repository already cloned. Pulling latest...")
    !cd {PROJECT_DIR} && git pull

os.chdir(os.path.join(PROJECT_DIR, "ml"))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Contents: {sorted(os.listdir('.'))}")


In [ ]:
# ============================================================
# Cell 3: Imports & Configuration
# ============================================================
import sys
import time
import json
import logging
import warnings
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for Colab inline display
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

# Suppress noisy warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="hmmlearn")

# Setup logging — show INFO level with timestamps
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("colab_pipeline")

# Ensure the ml/ package is importable
sys.path.insert(0, ".")

# Load configuration
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Create results directory
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

# Print configuration summary
print("=" * 60)
print("  CONFIGURATION LOADED")
print("=" * 60)
print(f"  Assets:      {config['data']['assets']}")
print(f"  Date range:  {config['data']['start_date']} to {config['data']['end_date']}")
print(f"  Frequency:   {config['data']['frequency']}")
print(f"  GARCH:       p={config['garch']['p']}, q={config['garch']['q']}, dist={config['garch']['distribution']}")
print(f"  HMM:         n_states={config['hmm']['n_states']}, state_selection={config['hmm']['enable_state_selection']}")
print(f"  RL:          algorithm={config['rl']['algorithm']}, timesteps={config['rl']['training']['total_timesteps']}")
print(f"               seeds={config['rl']['training']['n_seeds']}, top_k={config['rl']['training']['top_k_ensemble']}")
print(f"               curriculum={config['rl']['training']['enable_curriculum']}, lr_schedule={config['rl']['training']['enable_lr_schedule']}")
print(f"  Backtest:    train={config['backtest']['train_window']}h, test={config['backtest']['test_window']}h")
print(f"               TC={config['backtest']['transaction_cost_bps']} bps, MC sims={config['backtest']['monte_carlo_sims']}")
print(f"  Ensemble:    method={config['ensemble']['method']}, CVaR alpha={config['ensemble']['cvar_confidence']}")
print(f"               circuit_breaker_threshold={config['ensemble']['circuit_breaker']['drawdown_threshold']:.0%}")
print("=" * 60)

## Step 1: Data Fetching (~10 min)

Fetches hourly OHLCV data for all 8 portfolio constituents:
- **Crypto:** BTC, ETH, SOL (via Binance with Kraken/Coinbase fallback)
- **Liquid staking:** stETH, rETH (via Binance)
- **Tokenised treasuries:** BUIDL, USDY (synthetic GBM, calibrated to T-bill vol)
- **Stablecoin:** USDC (synthetic with historical depeg events: SVB, UST)

Data is cached as Parquet files to avoid re-fetching on subsequent runs.

In [ ]:
%%time
# ============================================================
# Cell 4: Data Fetching
# ============================================================
from data.fetch_data import fetch_all_data

t0 = time.time()
print("Fetching OHLCV data for all portfolio constituents...")
print("  Primary: Binance | Fallback: Kraken, Coinbase")
print("  Synthetic: BUIDL (GBM, 4.5% yield), USDY (GBM, 5.0% yield), USDC (depeg-aware)")
print()

raw_data = fetch_all_data("config.yaml")

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  DATA FETCH COMPLETE ({elapsed:.1f}s)")
print(f"{'=' * 60}")

# Quality report
print(f"\n{'Asset':<8} {'Rows':>8} {'Start':>12} {'End':>12}")
print("-" * 50)
for sym, df in raw_data.items():
    print(f"{sym:<8} {len(df):>8} {str(df.index[0])[:10]:>12} {str(df.index[-1])[:10]:>12}")

# Show sample data for BTC
print(f"\nBTC sample (last 5 rows):")
display(raw_data["BTC"].tail())

# Quick sanity chart: BTC close price
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(raw_data["BTC"].index, raw_data["BTC"]["close"], linewidth=0.5, color="#f7931a")
ax.set_title("BTC/USDT Hourly Close Price", fontsize=14, fontweight="bold")
ax.set_ylabel("Price (USDT)")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("Data fetch complete.")

## Step 2: Preprocessing

Full feature engineering pipeline:
- **Log returns** with outlier handling (winsorization at [1, 99] percentiles)
- **Stationarity tests** (ADF at 5% significance)
- **Volatility estimators:** Close-to-close, Parkinson (1980), Garman-Klass (1980)
- **Features:** Momentum, rolling Sharpe, rolling skew/kurtosis, Hurst exponent, Amihud illiquidity, BTC correlation
- **Cross-asset features:** Market breadth, average pairwise correlation, PCA variance ratio
- **HMM features:** 24h mean return, 5d realised vol, market breadth
- **Normalisation:** Rolling z-score (720h window)

In [ ]:
%%time
# ============================================================
# Cell 5: Preprocessing Pipeline
# ============================================================
from data.preprocess import (
    prepare_all_data,
    test_stationarity,
    compute_cross_asset_features,
)

t0 = time.time()
print("Running full preprocessing pipeline...")
print("  Outlier handling:  winsorize [1, 99] percentiles")
print("  Stationarity:      ADF test at 5% significance")
print("  Normalisation:     rolling z-score (720h window)")
print()

prices_df, returns_df, features_df, hmm_features = prepare_all_data("config.yaml")

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  PREPROCESSING COMPLETE ({elapsed:.1f}s)")
print(f"{'=' * 60}")
print(f"  Prices:       {prices_df.shape[0]:,} rows x {prices_df.shape[1]} assets")
print(f"  Returns:      {returns_df.shape[0]:,} rows x {returns_df.shape[1]} assets")
print(f"  Features:     {features_df.shape}")
print(f"  HMM features: {hmm_features.shape[0]:,} rows x {hmm_features.shape[1]} features")
print(f"  Date range:   {prices_df.index[0]} to {prices_df.index[-1]}")

# Display stationarity test results
print(f"\nStationarity Test Results (ADF, alpha=5%):")
stationarity_path = Path("data/cache/stationarity_report.json")
if stationarity_path.exists():
    with open(stationarity_path) as f:
        stat_results = json.load(f)
    stat_df = pd.DataFrame(stat_results).T[["statistic", "pvalue", "is_stationary"]]
    stat_df.columns = ["ADF Statistic", "p-value", "Stationary?"]
    display(stat_df)
else:
    print("  (Stationarity report not cached; see logs above)")

# Return statistics summary
print(f"\nReturn Statistics (hourly):")
ret_stats = returns_df.describe().T[["mean", "std", "min", "max"]]
ret_stats["annualised_vol"] = returns_df.std() * np.sqrt(8760)
display(ret_stats.round(6))

# Plot: normalised close prices
fig, ax = plt.subplots(figsize=(14, 6))
normalised = prices_df / prices_df.iloc[0]
for col in normalised.columns:
    ax.plot(normalised.index, normalised[col], label=col, linewidth=0.8)
ax.set_title("Normalised Close Prices (base = 1.0)", fontsize=14, fontweight="bold")
ax.set_ylabel("Normalised Price")
ax.legend(loc="upper left", fontsize=9, ncol=4)
ax.set_yscale("log")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
print("Preprocessing complete.")

## Step 3: GARCH-DCC Covariance Estimation (~5 min)

Fits a **Student-t GARCH(1,1)-DCC** model for time-varying covariance:

1. **Univariate volatility:** Selects best of GARCH(1,1) / EGARCH(1,1) / GJR-GARCH(1,1) per asset via BIC
2. **Diagnostics:** Ljung-Box (autocorrelation), ARCH-LM (heteroskedasticity), Jarque-Bera (normality)
3. **DCC correlation:** Quasi-ML estimation with method-of-moments initialisation
4. **Risk-parity weights:** Equal risk contribution from the forecast covariance matrix

In [ ]:
%%time
# ============================================================
# Cell 6: GARCH-DCC Fitting
# ============================================================
from models.garch_dcc import StudentTGarchDCC

t0 = time.time()
asset_names = list(returns_df.columns)
n_assets = len(asset_names)

print("Fitting Student-t GARCH-DCC covariance model...")
print(f"  Model selection: GARCH vs EGARCH vs GJR-GARCH (BIC)")
print(f"  Diagnostics: Ljung-Box, ARCH-LM, Jarque-Bera")
print()

garch = StudentTGarchDCC(
    p=config["garch"]["p"],
    q=config["garch"]["q"],
    distribution=config["garch"]["distribution"],
    config=config.get("garch", {}),
)
garch.fit(returns_df)

# Forecast covariance matrix
sigma = garch.forecast_covariance()
garch_rp_weights = garch.get_risk_parity_weights(sigma)
garch_uncertainty = garch.get_uncertainty()

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  GARCH-DCC COMPLETE ({elapsed:.1f}s)")
print(f"{'=' * 60}")

# Model selection results
print(f"\nModel Selection (best per asset by BIC):")
for asset, result in garch.model_selection_results.items():
    print(f"  {asset:<6}: {result['selected']:<12} (BIC delta: {result['bic_delta']})")

# Volatility type selected
print(f"\nSelected volatility models:")
for asset, vol_type in garch.selected_vol_type.items():
    print(f"  {asset:<6}: {vol_type}")

# Diagnostic test summary
print(f"\nDiagnostic Tests:")
print(f"  {'Asset':<8} {'Ljung-Box':>10} {'ARCH-LM':>10} {'Jarque-Bera':>12}")
print("  " + "-" * 44)
for asset, diag in garch.diagnostic_results.items():
    lb = "PASS" if diag.get("ljung_box", {}).get("pass") else "FAIL"
    lm = "PASS" if diag.get("arch_lm", {}).get("pass") else "FAIL"
    jb = "PASS" if diag.get("jarque_bera", {}).get("pass") else "FAIL"
    print(f"  {asset:<8} {lb:>10} {lm:>10} {jb:>12}")

# DCC parameters
print(f"\nDCC Parameters: a={garch.dcc_params['a']:.4f}, b={garch.dcc_params['b']:.4f}, "
      f"a+b={garch.dcc_params['a'] + garch.dcc_params['b']:.4f}")

# Risk-parity weights
print(f"\nGARCH Risk-Parity Weights:")
for a, w in zip(asset_names, garch_rp_weights):
    print(f"  {a:<6}: {w:.4f} ({w*100:.1f}%)")
print(f"  Uncertainty: {garch_uncertainty:.6f}")

# Plot: Covariance matrix heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation matrix
vols = np.sqrt(np.diag(sigma))
corr = sigma / np.outer(vols, vols)
sns.heatmap(pd.DataFrame(corr, index=asset_names, columns=asset_names),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=axes[0], square=True)
axes[0].set_title("DCC Forecast Correlation", fontsize=12, fontweight="bold")

# Annualised volatility bar chart
ann_vols = vols * np.sqrt(8760) * 100
axes[1].barh(asset_names, ann_vols, color="#3498db", alpha=0.8)
axes[1].set_xlabel("Annualised Volatility (%)")
axes[1].set_title("GARCH Forecast Volatility", fontsize=12, fontweight="bold")
axes[1].grid(True, alpha=0.3, axis="x")

fig.tight_layout()
fig.savefig(results_dir / "garch_dcc_diagnostics.png", dpi=300, bbox_inches="tight")
plt.show()
print("GARCH-DCC fitting complete.")

## Step 4: Bayesian HMM Regime Detection (~2 min)

Fits a **Gaussian Hidden Markov Model** for market regime classification:

1. **State selection:** Compares 2-5 states via BIC + time-series cross-validated log-likelihood
2. **Multiple restarts:** 10 random initialisations with sticky transition prior to avoid local optima
3. **State sorting:** States ordered by mean return (bull > normal > crisis)
4. **Outputs:** Soft posterior probabilities for continuous risk-budget blending

The soft posteriors (not hard labels) drive the ensemble risk budget:
- Bull: 70% crypto, 20% staking, 10% stable
- Normal: 40% crypto, 30% staking, 20% treasuries, 10% stable  
- Crisis: 10% crypto, 10% staking, 50% treasuries, 30% stable

In [ ]:
%%time
# ============================================================
# Cell 7: Bayesian HMM Fitting
# ============================================================
from models.bayesian_hmm import BayesianRegimeHMM

t0 = time.time()
print("Fitting Bayesian Gaussian HMM for regime detection...")
print(f"  State selection: comparing {config['hmm']['state_range']} states via BIC + CV")
print(f"  Random restarts: {config['hmm']['n_init']}")
print(f"  CV folds: {config['hmm']['cv_folds']}")
print()

hmm_model = BayesianRegimeHMM(
    n_states=config["hmm"]["n_states"],
    n_iter=config["hmm"]["n_iter"],
    covariance_type=config["hmm"]["covariance_type"],
    config=config.get("hmm", {}),
)
hmm_model.fit(hmm_features)

# Get regime probabilities for full dataset
regime_probs_df = hmm_model.predict_proba(hmm_features)
regime_labels = hmm_model.predict_regime(hmm_features)
latest_probs = regime_probs_df.iloc[-1].values

# Also get regime change probabilities
regime_change_df = hmm_model.predict_regime_change_probability(hmm_features)
hmm_uncertainty = hmm_model.get_uncertainty()

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  HMM FITTING COMPLETE ({elapsed:.1f}s)")
print(f"{'=' * 60}")

# State selection results
if hmm_model.state_selection_results:
    print(f"\nState Selection Results:")
    print(f"  {'States':>6} {'BIC':>12} {'CV Log-LL':>12} {'Selected':>10}")
    print("  " + "-" * 44)
    for n_s, res in sorted(hmm_model.state_selection_results.items()):
        marker = "  <--" if n_s == hmm_model.optimal_n_states else ""
        print(f"  {n_s:>6} {res['bic']:>12.1f} {res['cv_log_likelihood']:>12.1f} {marker}")

print(f"\n  Optimal states: {hmm_model.optimal_n_states}")
print(f"  Uncertainty: {hmm_uncertainty:.4f}")

# Transition matrix
print(f"\nTransition Matrix:")
trans_matrix = hmm_model.get_transition_matrix()
display(trans_matrix.round(3))

# Regime persistence
print(f"\nRegime Persistence:")
persistence = hmm_model.get_regime_persistence()
for label, stats in persistence.items():
    print(f"  {label:<8}: E[duration]={stats['expected_days']:.1f} days, "
          f"P(stay)={stats['self_transition_prob']:.3f}")

# Regime duration analysis
if hmm_model.regime_duration_stats:
    print(f"\nEmpirical Regime Durations:")
    for label, stats in hmm_model.regime_duration_stats.items():
        print(f"  {label:<8}: {stats['n_episodes']} episodes, "
              f"mean={stats['empirical_mean_days']:.1f}d, "
              f"median={stats['empirical_median_days']:.1f}d, "
              f"range=[{stats['empirical_min_hours']:.0f}h, {stats['empirical_max_hours']:.0f}h]")

# Regime-conditional statistics
if hmm_model.regime_conditional_stats:
    print(f"\nRegime-Conditional Statistics:")
    for label, stats in hmm_model.regime_conditional_stats.items():
        ret_str = f"ret_mean={stats.get('return_mean', 0):.6f}" if "return_mean" in stats else ""
        print(f"  {label:<8}: {stats['fraction']:.1%} of data, {ret_str}")

# OOS accuracy
if hmm_model.oos_accuracy is not None:
    print(f"\n  OOS regime agreement: {hmm_model.oos_accuracy:.1%}")

# Latest regime
print(f"\nLatest regime probabilities:")
for col, prob in zip(regime_probs_df.columns, latest_probs):
    print(f"  {col}: {prob:.4f}")
print(f"  Dominant: {regime_labels.iloc[-1]}")

# Plot: Regime timeline + probabilities
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [1, 3]})

# Top: regime timeline (coloured bands)
REGIME_COLOURS = {"bull": "#27ae60", "normal": "#f39c12", "crisis": "#e74c3c"}
for regime, colour in REGIME_COLOURS.items():
    mask = regime_labels == regime
    if mask.any():
        for i in range(len(mask)):
            if mask.iloc[i]:
                axes[0].axvspan(hmm_features.index[i],
                               hmm_features.index[min(i + 1, len(mask) - 1)],
                               alpha=0.6, color=colour, linewidth=0)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.6, label=r.capitalize())
                   for r, c in REGIME_COLOURS.items()]
axes[0].legend(handles=legend_elements, loc="upper right", fontsize=10)
axes[0].set_title("HMM Regime Classification", fontsize=14, fontweight="bold")
axes[0].set_yticks([])
axes[0].set_xlim(hmm_features.index[0], hmm_features.index[-1])

# Bottom: regime probabilities stacked area
prob_cols = regime_probs_df.columns.tolist()
colours_ordered = ["#27ae60", "#f39c12", "#e74c3c"][:len(prob_cols)]
axes[1].stackplot(regime_probs_df.index, *[regime_probs_df[c] for c in prob_cols],
                  labels=prob_cols, colors=colours_ordered, alpha=0.7)
axes[1].set_ylabel("Posterior Probability")
axes[1].set_title("Regime Posterior Probabilities (Soft)", fontsize=12, fontweight="bold")
axes[1].legend(loc="upper left", fontsize=9)
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(results_dir / "hmm_regime_analysis.png", dpi=300, bbox_inches="tight")
plt.show()
print("HMM fitting complete.")

## Step 5: SAC Deep RL Training (~3-5 hours) -- THE MAIN GPU TASK

This is the most compute-intensive step. Trains a **Soft Actor-Critic (SAC)** agent for portfolio allocation:

- **Environment:** Custom `PortfolioEnv` (Gymnasium) with 48-dim state space, 8-dim action space
- **State:** Per-asset returns/vol/BTC-corr, GARCH vol forecasts, HMM regime probs, portfolio state, Kelly fraction
- **Reward:** Composite: return + CVaR penalty + drawdown penalty + turnover penalty + risk-parity deviation + HHI concentration
- **Training:** 5 seeds x 500K timesteps each with:
  - Curriculum learning (Phase 1: simple reward, Phase 2: full reward)
  - Cosine annealing learning rate schedule
  - RunningMeanStd observation normalisation
  - EvalCallback with monitoring
- **Selection:** Best seed by annualised Sharpe ratio, top-3 ensemble

**Expected runtime on T4 GPU: 3-5 hours**

In [ ]:
%%time
# ============================================================
# Cell 8: SAC Training — GPU-Accelerated
# ============================================================
from models.sac_agent import SACAllocator
from environment.portfolio_env import PortfolioEnv
import torch

t0 = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"
rl_cfg = config.get("rl", {})
train_cfg = rl_cfg.get("training", {})

print("=" * 60)
print("  SAC DEEP RL TRAINING")
print("=" * 60)
print(f"  Device:         {device}")
print(f"  Seeds:          {train_cfg.get('n_seeds', 5)}")
print(f"  Timesteps/seed: {train_cfg.get('total_timesteps', 500000):,}")
print(f"  Batch size:     {train_cfg.get('batch_size', 256)}")
print(f"  Learning rate:  {train_cfg.get('learning_rate', 3e-4)}")
print(f"  Curriculum:     {train_cfg.get('enable_curriculum', True)}")
print(f"  LR schedule:    {train_cfg.get('enable_lr_schedule', True)}")
print(f"  Obs normalise:  {train_cfg.get('enable_obs_normalization', True)}")
print(f"  Top-K ensemble: {train_cfg.get('top_k_ensemble', 3)}")
print("=" * 60)
print()

# ── Prepare training data from GARCH and HMM outputs ──
# returns: (T, n_assets) — raw log returns aligned with common index
train_returns = returns_df.values.astype(np.float32)

# features: (T, n_features) — per-asset features for the environment
# The PortfolioEnv expects features with shape (T, n_assets * 3) for
# [returns_5d, returns_20d, vol_20d] per asset.
# We construct these from rolling statistics on the return matrix.
T_total = train_returns.shape[0]
n_a = train_returns.shape[1]

returns_5d = pd.DataFrame(train_returns, index=returns_df.index,
                          columns=returns_df.columns).rolling(120).sum().values
returns_20d = pd.DataFrame(train_returns, index=returns_df.index,
                           columns=returns_df.columns).rolling(480).sum().values
vol_20d = pd.DataFrame(train_returns, index=returns_df.index,
                        columns=returns_df.columns).rolling(480).std().values * np.sqrt(8760)

# Stack features: (T, n_assets * 3)
train_features = np.concatenate([
    np.nan_to_num(returns_5d, nan=0.0),
    np.nan_to_num(returns_20d, nan=0.0),
    np.nan_to_num(vol_20d, nan=0.0),
], axis=1).astype(np.float32)

print(f"Training data shapes:")
print(f"  Returns:     {train_returns.shape}")
print(f"  Features:    {train_features.shape}")

# regime_probs: (T, 3) — HMM posteriors aligned to returns index
# Align regime probs to returns index (fill missing with uniform)
regime_probs_aligned = np.full((T_total, 3), 1.0 / 3, dtype=np.float32)
common_idx = returns_df.index.intersection(regime_probs_df.index)
if len(common_idx) > 0:
    # Map each timestamp to its position in returns_df
    returns_idx_map = {ts: i for i, ts in enumerate(returns_df.index)}
    for ts in common_idx:
        if ts in returns_idx_map:
            i = returns_idx_map[ts]
            probs = regime_probs_df.loc[ts].values
            # Ensure we have exactly 3 probabilities (pad/truncate for non-3-state models)
            if len(probs) >= 3:
                regime_probs_aligned[i, :] = probs[:3]
            else:
                regime_probs_aligned[i, :len(probs)] = probs
                regime_probs_aligned[i, len(probs):] = (1.0 - probs.sum()) / (3 - len(probs))

print(f"  Regime probs: {regime_probs_aligned.shape} (aligned to returns)")

# garch_vols: (T, n_assets) — GARCH conditional volatilities
# Use the fitted conditional volatilities, aligned to returns index
garch_vols_aligned = np.full((T_total, n_a), 0.01, dtype=np.float32)
if garch.conditional_vols is not None:
    cv_idx = garch.conditional_vols.index
    common_garch = returns_df.index.intersection(cv_idx)
    for ts in common_garch:
        if ts in returns_idx_map:
            i = returns_idx_map[ts]
            garch_vols_aligned[i, :] = garch.conditional_vols.loc[ts].values.astype(np.float32)

print(f"  GARCH vols:  {garch_vols_aligned.shape} (aligned to returns)")
print()

# ── Initialise and train the SAC agent ──
sac_agent = SACAllocator(config)

print(f"Starting training at {time.strftime('%H:%M:%S')}...")
print(f"Expected completion: ~{train_cfg.get('n_seeds', 5) * 40}-{train_cfg.get('n_seeds', 5) * 60} minutes on T4 GPU")
print()

sac_agent.train(
    returns=train_returns,
    features=train_features,
    regime_probs=regime_probs_aligned,
    garch_vols=garch_vols_aligned,
    save_dir="models/saved",
)

elapsed = time.time() - t0
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)
seconds = int(elapsed % 60)

print(f"\n{'=' * 60}")
print(f"  SAC TRAINING COMPLETE ({hours}h {minutes}m {seconds}s)")
print(f"{'=' * 60}")

# Display seed metrics summary
print(f"\nSeed Performance Summary:")
print(f"  {'Seed':>4} {'Sharpe':>8} {'MaxDD':>8} {'CVaR5%':>8} {'Return':>8} {'Steps':>8}")
print("  " + "-" * 52)
for seed, metrics in sorted(sac_agent.seed_metrics.items(), key=lambda x: x[1]["sharpe"], reverse=True):
    print(f"  {seed:>4} {metrics['sharpe']:>8.3f} {metrics['max_drawdown']:>8.3f} "
          f"{metrics['cvar_5pct']:>8.4f} {metrics['total_return']:>8.4f} {metrics['n_steps']:>8}")

# Best seed info
best_idx = max(sac_agent.seed_metrics, key=lambda k: sac_agent.seed_metrics[k]["sharpe"])
print(f"\n  Best seed: {best_idx} (Sharpe = {sac_agent.seed_metrics[best_idx]['sharpe']:.3f})")
print(f"  Ensemble: top-{train_cfg.get('top_k_ensemble', 3)} seeds")
print(f"  Models saved to: models/saved/")

# Quick test inference
test_obs = np.zeros(sac_agent.config.get("rl", {}).get("state_dim", 48), dtype=np.float32)
test_weights = sac_agent.predict(test_obs)
ensemble_weights, ensemble_std = sac_agent.predict_ensemble(test_obs)
det_weights, uncertainty = sac_agent.predict_with_uncertainty(test_obs)

print(f"\nTest inference (zero observation):")
print(f"  Single model:  {dict(zip(asset_names, test_weights.round(4)))}")
print(f"  Ensemble mean: {dict(zip(asset_names, ensemble_weights.round(4)))}")
print(f"  Ensemble std:  {dict(zip(asset_names, ensemble_std.round(4)))}")
print(f"  Uncertainty:   {uncertainty:.4f}")

## Step 6: Walk-Forward Backtest (~1-2 hours)

Rigorous out-of-sample evaluation using rolling walk-forward protocol:

- **Train window:** 180 days (4,320 hours), **Test window:** 30 days (720 hours)
- **Roll step:** 30 days — refit GARCH-DCC and HMM each fold
- **SAC:** Inference only (pre-trained model from Step 5)
- **Transaction costs:** 10 bps per unit of turnover
- **Benchmarks:** Equal Weight (1/N), BTC Only, 60/40 BTC/USDC, Market-Cap Weighted

Generates: equity curves, drawdown plot, regime timeline, weight evolution, rolling Sharpe, monthly returns heatmap, risk dashboard

In [ ]:
%%time
# ============================================================
# Cell 9: Walk-Forward Backtest (with trained SAC agent)
# ============================================================
from backtest.walk_forward import (
    run_walk_forward,
    plot_monthly_returns_heatmap,
    plot_risk_metrics_dashboard,
)

t0 = time.time()
print("=" * 60)
print("  WALK-FORWARD BACKTEST (WITH SAC RL)")
print("=" * 60)
print(f"  Train window: {config['backtest']['train_window']}h ({config['backtest']['train_window']//24}d)")
print(f"  Test window:  {config['backtest']['test_window']}h ({config['backtest']['test_window']//24}d)")
print(f"  Step size:    {config['backtest']['step_size']}h ({config['backtest']['step_size']//24}d)")
print(f"  TC:           {config['backtest']['transaction_cost_bps']} bps")
print(f"  SAC agent:    ENABLED (models/saved/sac_best)")
print("=" * 60)
print()

# Run the full walk-forward backtest with the trained SAC agent
backtest_results = run_walk_forward(
    config_path="config.yaml",
    use_rl=True,
    rl_model_path="models/saved/sac_best",
)

elapsed = time.time() - t0
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print(f"\n{'=' * 60}")
print(f"  BACKTEST COMPLETE ({hours}h {minutes}m)")
print(f"{'=' * 60}")

# Display performance summary
print(f"\nPerformance Summary:")
metrics_summary = backtest_results["metrics"]
metrics_df = pd.DataFrame(metrics_summary).T

# Format key metrics for display
display_cols = [
    "annualised_return", "annualised_volatility", "sharpe_ratio",
    "sortino_ratio", "max_drawdown", "calmar_ratio",
    "cvar_5pct", "omega_ratio", "total_return",
]
available_cols = [c for c in display_cols if c in metrics_df.columns]
display(metrics_df[available_cols].round(4))

# Additional backtest charts: monthly returns heatmap and risk dashboard
try:
    plot_monthly_returns_heatmap(
        backtest_results["ensemble_returns"],
        backtest_results["timestamps"],
        results_dir,
    )
except Exception as e:
    print(f"  Monthly heatmap generation skipped: {e}")

try:
    plot_risk_metrics_dashboard(
        backtest_results["ensemble_returns"],
        backtest_results["timestamps"],
        results_dir,
    )
except Exception as e:
    print(f"  Risk dashboard generation skipped: {e}")

# Show all generated charts inline
chart_files = sorted(results_dir.glob("*.png"))
print(f"\nGenerated {len(chart_files)} charts:")
for cf in chart_files:
    print(f"  {cf.name}")

# Display key charts inline
from IPython.display import Image as IPImage

for chart_name in ["equity_curves.png", "drawdown_plot.png", "regime_timeline.png",
                    "weight_evolution.png", "rolling_sharpe.png",
                    "monthly_returns_heatmap.png", "risk_dashboard.png"]:
    chart_path = results_dir / chart_name
    if chart_path.exists():
        print(f"\n--- {chart_name} ---")
        display(IPImage(filename=str(chart_path), width=900))

## Step 7: Monte Carlo Stress Testing (~30 min)

**10,000-path** regime-conditioned block bootstrap simulation:

- Uses HMM transition matrix to sample regime sequences
- Draws 1-week (168h) return blocks from historical data conditional on regime
- Horizons: 6-month (4,320h) and 12-month (8,640h)
- Reports: fan charts, terminal NAV distribution, P(loss), P(Sharpe > 1), drawdown distribution

In [ ]:
%%time
# ============================================================
# Cell 10: Monte Carlo Stress Test
# ============================================================
from backtest.monte_carlo import run_monte_carlo

t0 = time.time()
print("=" * 60)
print("  MONTE CARLO STRESS TESTING")
print("=" * 60)
print(f"  Simulations: {config['backtest']['monte_carlo_sims']:,}")
print(f"  Block size:  168h (1 week)")
print(f"  Horizons:    6-month, 12-month")
print(f"  Method:      regime-conditioned block bootstrap")
print("=" * 60)
print()

# Prepare inputs for Monte Carlo
# Use the final backtest weights as the target allocation
if backtest_results.get("weights_history") and len(backtest_results["weights_history"]) > 0:
    mc_weights = np.array(backtest_results["weights_history"][-1])
else:
    mc_weights = garch_rp_weights

# Get regime labels aligned to returns for block sampling
regime_labels_arr = np.array(["normal"] * len(returns_df))
if len(regime_labels) > 0:
    common_mc = returns_df.index.intersection(regime_labels.index)
    for ts in common_mc:
        i = list(returns_df.index).index(ts) if ts in returns_df.index else -1
        if i >= 0:
            regime_labels_arr[i] = regime_labels.loc[ts]

# Get transition matrix as numpy array
trans_np = trans_matrix.values

print(f"Monte Carlo weights: {dict(zip(asset_names, mc_weights.round(4)))}")
print(f"Transition matrix:\n{trans_matrix.round(3).to_string()}")
print()

mc_results = run_monte_carlo(
    returns_df=returns_df,
    regimes=regime_labels_arr,
    transition_matrix=trans_np,
    weights=mc_weights,
    config=config,
)

elapsed = time.time() - t0
minutes = int(elapsed // 60)

print(f"\n{'=' * 60}")
print(f"  MONTE CARLO COMPLETE ({minutes}m)")
print(f"{'=' * 60}")

# Display statistics
for horizon_label, stats in mc_results.items():
    print(f"\n  {horizon_label} Horizon:")
    print(f"    Terminal value: median=${stats['terminal_value']['median']:.3f}, "
          f"5th=${stats['terminal_value']['p5']:.3f}, "
          f"95th=${stats['terminal_value']['p95']:.3f}")
    print(f"    Max drawdown:  median={stats['max_drawdown']['median']:.1%}")
    print(f"    Probabilities:")
    for prob_name, prob_val in stats["probabilities"].items():
        print(f"      {prob_name}: {prob_val:.1%}")

# Display MC charts inline
from IPython.display import Image as IPImage

for chart_name in ["monte_carlo_fan_6m.png", "terminal_dist_6m.png",
                    "monte_carlo_fan_12m.png", "terminal_dist_12m.png"]:
    chart_path = results_dir / chart_name
    if chart_path.exists():
        print(f"\n--- {chart_name} ---")
        display(IPImage(filename=str(chart_path), width=900))

## Step 8: Compute Final Weights for On-Chain Publication

Runs the **full ensemble pipeline** with all three models to produce the latest weight vector:

1. GARCH-DCC risk-parity weights (covariance-driven)
2. HMM regime-blended risk budget (soft posterior probabilities)
3. SAC RL agent weights (GPU-trained policy)
4. Inverse-variance combination + Kelly confidence scaling + CVaR constraint
5. Merkle root computation for on-chain commitment

The weight vector flows on-chain via: `commitWeights(merkleRoot)` -> 1hr timelock -> `executeWeights(weights[], proof[])`

In [ ]:
%%time
# ============================================================
# Cell 11: Compute Final Weights (Full Ensemble)
# ============================================================
from models.ensemble import EnsembleCombiner
from weight_publisher.merkle import compute_merkle_root

t0 = time.time()
print("=" * 60)
print("  FINAL WEIGHT COMPUTATION")
print("=" * 60)
print()

# ── 1. GARCH-DCC weights (already computed above) ──
print("1. GARCH-DCC risk-parity weights:")
for a, w in zip(asset_names, garch_rp_weights):
    print(f"     {a:<6}: {w:.4f}")
print(f"     Uncertainty: {garch_uncertainty:.6f}")

# ── 2. HMM regime probabilities (latest timestep) ──
print(f"\n2. HMM regime probabilities (latest):")
for col, prob in zip(regime_probs_df.columns, latest_probs):
    print(f"     {col}: {prob:.4f}")
print(f"     Uncertainty: {hmm_uncertainty:.4f}")

# ── 3. SAC RL weights ──
# Build a proper observation from the latest market state
latest_t = len(returns_df) - 1
latest_returns_5d = returns_df.iloc[max(0, latest_t - 120):latest_t].sum().values
latest_returns_20d = returns_df.iloc[max(0, latest_t - 480):latest_t].sum().values
latest_vol_20d = returns_df.iloc[max(0, latest_t - 480):latest_t].std().values * np.sqrt(8760)
latest_garch_vol = garch_vols_aligned[min(latest_t, len(garch_vols_aligned) - 1)]

# BTC correlation (rolling 7d)
btc_corr_latest = np.zeros(n_assets)
btc_ret = returns_df.iloc[max(0, latest_t - 168):latest_t]["BTC"].values
if len(btc_ret) > 10:
    for j, col in enumerate(asset_names):
        asset_ret = returns_df.iloc[max(0, latest_t - 168):latest_t][col].values
        if np.std(btc_ret) > 1e-10 and np.std(asset_ret) > 1e-10:
            btc_corr_latest[j] = np.corrcoef(btc_ret, asset_ret)[0, 1]

# Build 48-dim observation
latest_obs = np.concatenate([
    np.nan_to_num(latest_returns_5d, nan=0.0),    # 8: returns_5d
    np.nan_to_num(latest_returns_20d, nan=0.0),    # 8: returns_20d
    np.nan_to_num(latest_vol_20d, nan=0.0),         # 8: vol_20d
    np.nan_to_num(latest_garch_vol, nan=0.01),      # 8: GARCH vol
    np.nan_to_num(btc_corr_latest, nan=0.0),        # 8: BTC correlation
    latest_probs[:3],                                # 3: regime probs
    np.array([0.5]),                                 # 1: transition prob
    np.array([0.0, 0.0, 0.0, 0.5]),                # 4: cum_ret, drawdown, days_since_rebal, kelly
]).astype(np.float32)

# Pad/truncate to state_dim
state_dim = config.get("rl", {}).get("state_dim", 48)
if len(latest_obs) < state_dim:
    latest_obs = np.pad(latest_obs, (0, state_dim - len(latest_obs)))
elif len(latest_obs) > state_dim:
    latest_obs = latest_obs[:state_dim]

rl_weights = sac_agent.predict(latest_obs)
rl_ensemble_weights, rl_ensemble_std = sac_agent.predict_ensemble(latest_obs)
_, rl_uncertainty_score = sac_agent.predict_with_uncertainty(latest_obs)
rl_uncertainty = sac_agent.get_uncertainty()

print(f"\n3. SAC RL weights (ensemble of top-{train_cfg.get('top_k_ensemble', 3)} seeds):")
for a, w, s in zip(asset_names, rl_ensemble_weights, rl_ensemble_std):
    print(f"     {a:<6}: {w:.4f} (std: {s:.4f})")
print(f"     Uncertainty: {rl_uncertainty:.4f}")
print(f"     Prediction uncertainty: {rl_uncertainty_score:.4f}")

# ── 4. Ensemble combination ──
print(f"\n4. Running ensemble combiner...")
ensemble = EnsembleCombiner(config=config, asset_names=asset_names)
current_weights_init = np.ones(n_assets) / n_assets

ensemble_result = ensemble.combine(
    garch_rp_weights=garch_rp_weights,
    hmm_regime_probs=latest_probs[:3] if len(latest_probs) >= 3 else np.array([0.33, 0.34, 0.33]),
    rl_weights=rl_ensemble_weights,
    covariance_matrix=sigma,
    current_weights=current_weights_init,
    current_nav=1.0,
    garch_uncertainty=garch_uncertainty,
    hmm_uncertainty=hmm_uncertainty,
    rl_uncertainty=rl_uncertainty,
)

final_weights = ensemble_result["weights"]
weights_dict = {a: round(float(w), 6) for a, w in zip(asset_names, final_weights)}
weights_bps = {a: int(w * 10000) for a, w in weights_dict.items()}

print(f"\n   Model contributions:")
for model, contrib in ensemble_result["model_contributions"].items():
    print(f"     {model:<6}: {contrib:.4f} ({contrib*100:.1f}%)")

print(f"\n   Regime: {ensemble_result['regime']}")
print(f"   Circuit breaker: {ensemble_result['circuit_breaker']}")
if "kelly_scale" in ensemble_result:
    print(f"   Kelly scale: {ensemble_result['kelly_scale']:.4f}")
if "confidence" in ensemble_result:
    print(f"   Confidence: {ensemble_result['confidence']:.4f}")

# ── 5. Merkle root ──
print(f"\n5. Computing Merkle root for on-chain commitment...")
placeholder_addresses = [f"0x{'0' * 39}{i}" for i in range(n_assets)]
bps_list = [weights_bps[a] for a in asset_names]
merkle_root = compute_merkle_root(placeholder_addresses, bps_list)
print(f"   Merkle root: {merkle_root}")

# ── Display final weights ──
print(f"\n{'=' * 60}")
print(f"  FINAL PORTFOLIO WEIGHTS")
print(f"{'=' * 60}")
print(f"  {'Asset':<8} {'Weight':>8} {'BPS':>6} {'Category':<12}")
print("  " + "-" * 38)

categories = {
    "BTC": "Crypto", "ETH": "Crypto", "SOL": "Crypto",
    "stETH": "Staking", "rETH": "Staking",
    "BUIDL": "Treasury", "USDY": "Treasury",
    "USDC": "Stablecoin",
}

for a in asset_names:
    w = weights_dict[a]
    bps = weights_bps[a]
    cat = categories.get(a, "Other")
    print(f"  {a:<8} {w:>8.4f} {bps:>6} {cat:<12}")

print(f"  {'':─<38}")
print(f"  {'TOTAL':<8} {sum(final_weights):>8.4f} {sum(bps_list):>6}")
print(f"\n  Merkle root: {merkle_root}")

# Weight allocation pie chart
fig, ax = plt.subplots(figsize=(8, 8))
colors_pie = ["#f7931a", "#627eea", "#9945ff", "#00d4aa", "#ff6b35",
              "#2563eb", "#16a34a", "#6366f1"]
wedges, texts, autotexts = ax.pie(
    final_weights, labels=asset_names, autopct="%1.1f%%",
    colors=colors_pie[:n_assets], startangle=90, pctdistance=0.75,
)
for autotext in autotexts:
    autotext.set_fontsize(10)
ax.set_title("Final Portfolio Allocation", fontsize=16, fontweight="bold", pad=20)
fig.tight_layout()
fig.savefig(results_dir / "final_allocation.png", dpi=300, bbox_inches="tight")
plt.show()

# Save final weights to JSON
output = {
    "weights": weights_dict,
    "weights_bps": weights_bps,
    "merkle_root": merkle_root,
    "regime": ensemble_result["regime"],
    "regime_probs": ensemble_result.get("regime_probs", {}),
    "circuit_breaker": ensemble_result["circuit_breaker"],
    "model_contributions": ensemble_result["model_contributions"],
}
with open(results_dir / "latest_weights.json", "w") as f:
    json.dump(output, f, indent=2, default=str)
print(f"\nSaved to results/latest_weights.json")

## Results Summary

All pipeline outputs below. The weight vector and Merkle root are ready for on-chain publication via the keeper bot.

In [ ]:
# ============================================================
# Cell 12: Results Summary
# ============================================================
from IPython.display import HTML

# ── Backtest key metrics ──
ensemble_metrics = backtest_results["metrics"].get("Ensemble (Ours)", {})

summary_html = f"""
<style>
    .summary-table {{ border-collapse: collapse; width: 100%; font-family: monospace; }}
    .summary-table th {{ background-color: #1a1a2e; color: white; padding: 10px; text-align: left; }}
    .summary-table td {{ padding: 8px; border-bottom: 1px solid #ddd; }}
    .summary-table tr:nth-child(even) {{ background-color: #f9f9f9; }}
    .section-header {{ background-color: #16213e !important; color: #00ff88 !important; font-weight: bold; }}
</style>

<h2>Pipeline Results Summary</h2>

<table class="summary-table">
<tr><th colspan="2" class="section-header">Backtest Performance (Walk-Forward, OOS)</th></tr>
<tr><td>Annualised Return</td><td><b>{ensemble_metrics.get('annualised_return', 0):.2%}</b></td></tr>
<tr><td>Annualised Volatility</td><td>{ensemble_metrics.get('annualised_volatility', 0):.2%}</td></tr>
<tr><td>Sharpe Ratio</td><td><b>{ensemble_metrics.get('sharpe_ratio', 0):.3f}</b></td></tr>
<tr><td>Sortino Ratio</td><td>{ensemble_metrics.get('sortino_ratio', 0):.3f}</td></tr>
<tr><td>Max Drawdown</td><td>{ensemble_metrics.get('max_drawdown', 0):.2%}</td></tr>
<tr><td>Calmar Ratio</td><td>{ensemble_metrics.get('calmar_ratio', 0):.3f}</td></tr>
<tr><td>CVaR (5%)</td><td>{ensemble_metrics.get('cvar_5pct', 0):.4%}</td></tr>
<tr><td>Omega Ratio</td><td>{ensemble_metrics.get('omega_ratio', 0):.3f}</td></tr>
<tr><td>Total Return</td><td><b>{ensemble_metrics.get('total_return', 0):.2%}</b></td></tr>

<tr><th colspan="2" class="section-header">Model Configuration</th></tr>
<tr><td>GARCH-DCC</td><td>Student-t, model selection (GARCH/EGARCH/GJR per asset)</td></tr>
<tr><td>HMM States</td><td>{hmm_model.optimal_n_states} (selected via BIC from {config['hmm']['state_range']})</td></tr>
<tr><td>SAC RL</td><td>{train_cfg.get('n_seeds', 5)} seeds x {train_cfg.get('total_timesteps', 500000):,} steps, curriculum + cosine LR</td></tr>
<tr><td>Ensemble</td><td>Inverse-variance + Kelly scaling + CVaR constraint</td></tr>

<tr><th colspan="2" class="section-header">Current Regime</th></tr>
<tr><td>Dominant Regime</td><td><b>{ensemble_result['regime'].upper()}</b></td></tr>
<tr><td>P(bull)</td><td>{latest_probs[0]:.4f}</td></tr>
<tr><td>P(normal)</td><td>{latest_probs[1]:.4f}</td></tr>
<tr><td>P(crisis)</td><td>{latest_probs[2] if len(latest_probs) > 2 else 0:.4f}</td></tr>
<tr><td>Circuit Breaker</td><td>{'ACTIVE' if ensemble_result['circuit_breaker'] else 'Inactive'}</td></tr>

<tr><th colspan="2" class="section-header">On-Chain Publication</th></tr>
<tr><td>Merkle Root</td><td><code>{merkle_root}</code></td></tr>
<tr><td>Weight Vector (BPS)</td><td><code>{weights_bps}</code></td></tr>
</table>
"""

display(HTML(summary_html))

# Full metrics comparison table
print("\nFull Performance Comparison (all strategies):")
all_metrics_df = pd.DataFrame(backtest_results["metrics"]).T
display(all_metrics_df.round(4))

## Step 9: Download Results

Pack all results (charts, CSVs, trained models, weight files) into a zip for download.

**VS Code Colab Extension note:** `google.colab.files.download()` does NOT work when
running via the VS Code Colab extension (it only works in the browser). Instead, use
**Step 10** below to push results directly to GitHub from the Colab runtime, then pull
them to your local machine.

In [ ]:
# ============================================================
# Cell 13: Download Results
# ============================================================
import shutil

# List all result files
print("Results directory contents:")
all_result_files = sorted(results_dir.glob("*"))
for rf in all_result_files:
    size_kb = rf.stat().st_size / 1024
    print(f"  {rf.name:<40} {size_kb:>8.1f} KB")

# List saved models
models_dir = Path("models/saved")
if models_dir.exists():
    print(f"\nSaved models:")
    for mf in sorted(models_dir.glob("*")):
        size_kb = mf.stat().st_size / 1024
        print(f"  {mf.name:<40} {size_kb:>8.1f} KB")

# Create zip archive with results and models
zip_path = "/content/pipeline_results"
print(f"\nPacking results into {zip_path}.zip ...")

# Create a temporary directory with everything
staging = Path("/content/pipeline_staging")
staging.mkdir(parents=True, exist_ok=True)

# Copy results
if results_dir.exists():
    shutil.copytree(results_dir, staging / "results", dirs_exist_ok=True)

# Copy saved models
if models_dir.exists():
    shutil.copytree(models_dir, staging / "models_saved", dirs_exist_ok=True)

# Copy data quality report
quality_report = Path("data/cache/data_quality_report.json")
if quality_report.exists():
    shutil.copy2(quality_report, staging / "data_quality_report.json")

stationarity_report = Path("data/cache/stationarity_report.json")
if stationarity_report.exists():
    shutil.copy2(stationarity_report, staging / "stationarity_report.json")

# Create the zip
shutil.make_archive(zip_path, "zip", staging)

# Clean up staging
shutil.rmtree(staging)

zip_size_mb = Path(f"{zip_path}.zip").stat().st_size / (1024 * 1024)
print(f"Archive size: {zip_size_mb:.1f} MB")

# Attempt browser download (only works in browser Colab, NOT VS Code extension)
try:
    from google.colab import files
    print("\nAttempting browser download...")
    files.download(f"{zip_path}.zip")
    print("Download initiated. Check your browser downloads.")
except ImportError:
    print(f"\nNot running in browser Colab. Results saved to: {zip_path}.zip")
except Exception as e:
    print(f"\nBrowser download not available (expected when using VS Code Colab extension).")
    print(f"  Use Step 10 below to push results to GitHub instead.")
    print(f"  Results archive is at: {zip_path}.zip")

print("\n" + "=" * 60)
print("  RESULTS PACKED")
print("=" * 60)
print(f"  All results saved to: {results_dir}")
print(f"  Trained models at:    {models_dir}")
print(f"  Archive at:           {zip_path}.zip")
print("\n  >>> Run Step 10 below to push results to GitHub <<<")
print("=" * 60)

## Step 10: Push Results to GitHub

Since the VS Code Colab extension runs code on Google's remote runtime,
`files.download()` does not work. Instead, we push results directly to GitHub
from the Colab runtime.

**Authentication:** The cell below uses a GitHub Personal Access Token (PAT).
You can create one at https://github.com/settings/tokens (scope: `repo`).
The token is passed via `getpass` so it is never stored in the notebook.

**Alternative:** If you prefer not to authenticate, skip this cell and instead:
1. From the Colab terminal, run `!cp /content/pipeline_results.zip /content/drive/MyDrive/` (if Drive is mounted)
2. Or download the zip from the Colab file browser in your web browser

In [ ]:
# ============================================================
# Cell 14: Push Results to GitHub
# ============================================================
import subprocess, getpass, os

os.chdir(PROJECT_DIR)  # Back to repo root
print(f"Working directory: {os.getcwd()}")

# Configure git identity for the commit
subprocess.run(["git", "config", "user.email", "colab@pipeline.local"], cwd=PROJECT_DIR)
subprocess.run(["git", "config", "user.name", "Colab Pipeline"], cwd=PROJECT_DIR)

# Stage results, saved models, and weight files
print("\nStaging files...")
subprocess.run(["git", "add", "ml/results/"], cwd=PROJECT_DIR)
subprocess.run(["git", "add", "ml/models/saved/"], cwd=PROJECT_DIR)

# Show what will be committed
status = subprocess.run(["git", "status", "--short"], cwd=PROJECT_DIR, capture_output=True, text=True)
print(f"\nFiles to commit:")
print(status.stdout)

if not status.stdout.strip():
    print("Nothing to commit (results may already be up to date).")
else:
    # Commit
    result = subprocess.run(
        ["git", "commit", "-m", "Add ML pipeline results: backtest, Monte Carlo, trained SAC models, final weights"],
        cwd=PROJECT_DIR, capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"Commit error: {result.stderr}")
    else:
        # Push — requires authentication
        print("\nTo push to GitHub, enter your Personal Access Token (PAT).")
        print("Create one at: https://github.com/settings/tokens (scope: repo)")
        print("Leave blank to skip push (you can push manually later).\n")
        token = getpass.getpass("GitHub PAT (or Enter to skip): ")

        if token.strip():
            # Construct authenticated remote URL
            auth_url = f"https://{token.strip()}@github.com/abailey81/Regime-Switching-Risk-Parity-Crypto-Index-Vault.git"
            push_result = subprocess.run(
                ["git", "push", auth_url, "main"],
                cwd=PROJECT_DIR, capture_output=True, text=True,
            )
            if push_result.returncode == 0:
                print("Push successful!")
                print(push_result.stdout)
            else:
                print(f"Push failed: {push_result.stderr}")
                print("You can push manually from your local machine:")
                print("  cd /path/to/Regime-Switching-Risk-Parity-Crypto-Index-Vault")
                print("  git pull origin main && git push origin main")
        else:
            print("\nPush skipped. To get results to your local machine:")
            print("  1. On your local machine, run: git pull origin main")
            print("  2. Or download pipeline_results.zip from Colab file browser")

# Switch back to ml/ directory for any further work
os.chdir(os.path.join(PROJECT_DIR, "ml"))

print("\n" + "=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)
print(f"  Results:  {PROJECT_DIR}/ml/results/")
print(f"  Models:   {PROJECT_DIR}/ml/models/saved/")
print(f"  Archive:  /content/pipeline_results.zip")
print("=" * 60)